# 2.0 · Análisis del modelo campeón

Notebook **solo de exploración**: toda la lógica vive en el paquete `mibici` y aquí únicamente se importa.
Requiere haber corrido `make features` y `make tune` (o `make pipeline`).

- Datos: `demanda-v4` (`data/processed/model_input.parquet`), partición de prueba = septiembre 2026.
- Modelo: alias `champion` del Model Registry `mibici-demanda-estacion-hora`.

In [1]:
import pandas as pd
from mlflow.tracking import MlflowClient

from mibici import config
from mibici.modeling.predict import load_model, predict
from mibici.modeling.train import evaluate, top_stations
from mibici.plots import plot_error_by_hour, plot_forecast_week

pd.set_option("display.float_format", "{:.3f}".format)

## 1. Versiones registradas en el Model Registry

In [2]:
client = MlflowClient(tracking_uri=config.MLFLOW_TRACKING_URI)
versions = client.search_model_versions(f"name='{config.REGISTERED_MODEL}'")
champion = client.get_model_version_by_alias(config.REGISTERED_MODEL, "champion").version
pd.DataFrame([{
    "versión": v.version,
    "modelo": v.tags.get("modelo"),
    "alias": "champion" if v.version == champion else "",
    "devianza valid": float(v.tags["devianza_poisson_valid"]),
    "MAE valid": float(v.tags["mae_valid"]),
    "MAE test": float(v.tags["mae_test"]),
    "MAE test top 20 %": float(v.tags["mae_top_estaciones_test"]),
    "sesgo test": float(v.tags["sesgo_test"]),
} for v in versions]).sort_values("versión")

,versión,modelo,alias,devianza valid,MAE valid,MAE test,MAE test top 20 %,sesgo test
2,1,poisson_glm,,1.778,1.207,1.342,2.540,-0.120
1,2,hgb_poisson,champion,1.099,0.865,1.005,1.896,-0.228
0,3,random_forest,,1.161,0.902,1.042,2.005,-0.150


## 2. Pronóstico de septiembre 2026 con el campeón y comparación con la persistencia semanal

In [3]:
df = pd.read_parquet(config.PROCESSED_DATA_DIR / "model_input.parquet")
train = df[df["particion"] == "train"]
test = df[df["particion"] == "test"].reset_index(drop=True)
model = load_model("champion")
test["pred"] = predict(model, test)

top = test["estacion_id"].isin(top_stations(train))  # 20 % con más salidas en entrenamiento
pd.DataFrame({
    "campeón": evaluate(test["salidas"], test["pred"], top),
    "persistencia semanal": evaluate(test["salidas"], test["lag_168h"], top),
}).T

,mae,rmse,devianza_poisson,sesgo,mae_top_estaciones
campeón,1.005,1.712,1.281,-0.228,1.896
persistencia semanal,1.169,2.171,8.141,-0.140,2.375


**Lectura.** Septiembre es el mes de mayor demanda del año (hallazgo de Daniel en la Fase 1), así que conviene
revisar el sesgo: un valor negativo indica que el modelo subestima el nivel de demanda del mes.

In [4]:
fig = plot_forecast_week(test, test["pred"])
fig.savefig(config.FIGURES_DIR / "fig9_pronostico_semana.png", bbox_inches="tight")

In [5]:
fig = plot_error_by_hour(test, {"campeón": test["pred"], "persistencia semanal": test["lag_168h"]})
fig.savefig(config.FIGURES_DIR / "fig10_mae_por_hora.png", bbox_inches="tight")

## 3. Error por estación: ¿dónde se equivoca más?

In [6]:
by_station = (test.assign(error=(test["salidas"] - test["pred"]).abs())
              .groupby("estacion_id").agg(salidas_media=("salidas", "mean"), mae=("error", "mean")))
by_station.sort_values("mae", ascending=False).head(10)

,salidas_media,mae
estacion_id,,
51,17.038,5.749
35,12.406,4.526
271,11.568,3.668
50,9.263,3.570
54,8.307,3.177
194,6.212,3.173
11,9.590,2.926
34,5.985,2.781
52,8.275,2.728


**Lectura.** El error absoluto crece con el volumen de la estación: las estaciones más concurridas concentran
el error en salidas, aunque en términos relativos el modelo les va mejor. Por eso se reporta `mae_top_estaciones` además del MAE global.